# MERCURY preprocessor
This preprocessing tool is not part of the MERCURY pipeline, but is intended as a support tool to convert datasets into a MERCURY-compatible data format. At a minimum, MERCURY requires the sentence and span ranges of one entity per sentence. If a sentence contains multiple entities, the sentence ought to be copied multiple times. Sentences without entities shouldn't be in the final output. You can include optional columns such as source_file_name to be able to find the original source of the file easily at a later time.

In [1]:
import sentencex   # https://diff.wikimedia.org/2023/10/23/sentencex-empowering-nlp-with-multilingual-sentence-extraction/
from transformers import AutoTokenizer, AutoModelForTokenClassification 
import os
import sys
import csv
sys.path.append('utilities')

In [2]:
import gpu_manager

In [3]:
max_gpus = 1
gpu_manager.pick_gpu(1, 'auto', int(max_gpus))
os.environ["TOKENIZERS_PARALLELISM"] = "true"
devices = gpu_manager.pick_gpu(mode='report', verbosity=0)
if not bool(devices): 
    device = 'cpu'
else:
    device = 'cuda'

## Config
Setting up the support tool:

In [4]:
RAWFOLDER = "data/RAW data"
INCLUDE_FILENAME = True
INCLUDE_SPACY_ET_TYPE = True
OUTPUT_FILE = 'data/prepared_data.tsv'

## Entity extraction:
Use XLMRoberta trained on ENR tasks to prepare each extracted sentence for the final MERCURY pipeline.

In [ ]:
import torch

model_name = "julian-schelb/roberta-ner-multilingual"
tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(model_name).to(device)

def get_entities(tokenizer, model, sentence, device="cpu"):
    """Extract entity spans, character offsets, and entity types from a sentence.

    Returns:
        List of tuples: (start_idx, end_idx, entity_type, entity_text)
    """
    if not sentence or not sentence.strip():
        return []

    inputs = tokenizer(
        sentence,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
    )
    offset_mapping = inputs.pop("offset_mapping")[0].cpu().numpy()
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    predictions = logits.argmax(-1)[0].cpu().numpy()
    id2label = model.config.id2label

    entities = []
    current_entity = None

    for idx, pred_id in enumerate(predictions):
        label = id2label[pred_id]
        start_char, end_char = offset_mapping[idx]

        if start_char == end_char:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue

        if label != "O":
            tag_type = label.split("-")[-1] if "-" in label else label

            if current_entity is None:
                current_entity = {
                    "start": int(start_char),
                    "end": int(end_char),
                    "type": tag_type,
                }
            else:
                if tag_type == current_entity["type"] or label.startswith("I-"):
                    current_entity["end"] = int(end_char)
                else:
                    entities.append(current_entity)
                    current_entity = {
                        "start": int(start_char),
                        "end": int(end_char),
                        "type": tag_type,
                    }
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    result = []
    for ent in entities:
        s_idx = ent["start"]
        e_idx = ent["end"]
        ent_type = ent["type"]
        ent_text = sentence[s_idx:e_idx]
        if ent_text.strip():
            result.append((s_idx, e_idx, ent_type, ent_text))

    return result

## Extraction pipeline:
Process one document at a time and pipe the results into an output file.

In [ ]:
results_file = open(OUTPUT_FILE, 'w+', newline='', encoding='utf8')
writer = csv.writer(results_file, delimiter='\t')
writer.writerow(["clean_mention_sentence", "clean_entity_mention", "mention_start", "mention_end", "entity_type", "source_file"])

if os.path.exists(RAWFOLDER):
    for folder, subfolders, files in os.walk(RAWFOLDER):
        for file in files:
            filePath = os.path.join(folder, file)
            with open(filePath, "r", encoding="utf8") as text:
                text = text.replace('\n', ' ')
                text = text.replace('\r', ' ')
                text = text.replace ('  ', ' ')
                text = text.strip()
                sentences = sentencex.segment("xx", text.read())
                for sentence in sentences:
                    sentence = sentence.strip()
                    if not sentence:
                        continue
                    extracted_ets = get_entities(tokenizer, model, sentence, device=device)
                    for start_idx, end_idx, et_type, et_text in extracted_ets:
                        writer.writerow([sentence, et_text, start_idx, end_idx, et_type, file])

results_file.close()
print(f"[INFO] Preprocessing pipeline finished. Results written to '{OUTPUT_FILE}'.")